# Anomaly Detection Pipeline

Demonstrates all five anomaly detection models:
- Adaptive Baseline Detector (p99.9 + multi-signal)
- Correlated Degradation Detector
- Latency Shift Detector (CUSUM)
- Jitter Diagnosis Matrix (root cause classification)
- Outage Type Classifier

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from inventor_analysis.loaders import load_ping_data
from inventor_analysis.anomaly_detection import (
    AdaptiveBaselineDetector,
    CorrelatedDegradationDetector,
    LatencyShiftDetector,
    JitterDiagnosisMatrix,
    OutageClassifier,
)

## Load Data and Train/Test Split

In [ ]:
df = load_ping_data("../sample_data", max_files=5)

split_point = int(len(df) * 0.7)
train_data = df.iloc[:split_point].copy()
test_data = df.iloc[split_point:].copy()

print(f"Loaded {len(df):,} records, {df['target_host'].nunique()} hosts")
print(f"Train: {len(train_data):,}, Test: {len(test_data):,}")

## 1. Adaptive Baseline Detector

Uses p99.9 thresholds with multi-signal verification (RTT + jitter or packet loss). Designed for high precision — anomalies that are flagged are almost always real.

In [ ]:
detector = AdaptiveBaselineDetector()
detector.train(train_data)

predictions, scores, reasons = detector.predict(test_data)
anomaly_count = predictions.sum()

print(f"Anomalies detected: {anomaly_count} ({anomaly_count/len(test_data)*100:.2f}%)")

if anomaly_count > 0:
    anomaly_df = test_data[predictions].copy()
    anomaly_df['reason'] = [r for r, p in zip(reasons, predictions) if p]
    anomaly_df['severity'] = scores[predictions]
    print("\nTop anomalies:")
    display(anomaly_df.nlargest(5, 'rtt_avg')[
        ['timestamp', 'target_host', 'rtt_avg', 'jitter', 'pkts_lost', 'reason']
    ])

## 2. Correlated Degradation Detector

Detects network-wide incidents when multiple destinations degrade within the same time window.

In [ ]:
corr_detector = CorrelatedDegradationDetector(time_window_minutes=5, min_affected=3)
incidents = corr_detector.detect(test_data, detector)

if not incidents.empty:
    print(f"Network-wide incidents: {len(incidents)}")
    display(incidents.head())
else:
    print("No correlated incidents detected (network stable)")

## 3. Latency Shift Detector (CUSUM)

Detects sustained latency level changes (e.g., baseline moving from 5ms to 15ms) rather than transient spikes.

In [ ]:
shift_detector = LatencyShiftDetector(sensitivity=5.0, drift=1.0)
shifts = shift_detector.detect(df)

if not shifts.empty:
    print(f"Latency shifts detected: {len(shifts)}")
    print("\nTop shifts by RTT change:")
    display(shifts.nlargest(5, 'rtt_change_pct')[
        ['timestamp', 'destination', 'baseline_rtt', 'new_rtt', 'rtt_change_pct']
    ])
else:
    print("No significant latency shifts detected")

## 4. Jitter Diagnosis Matrix

Classifies the root cause of degradation using concurrent jitter, RTT, and packet loss:

| Condition | Diagnosis |
|---|---|
| High jitter + normal RTT | `CONGESTION` |
| Normal jitter + high RTT | `PATH_CHANGE_OR_ROUTING` |
| High jitter + high RTT | `LINK_SATURATION` |
| All degraded | `INFRASTRUCTURE_FAILURE` |

In [ ]:
diagnoser = JitterDiagnosisMatrix(percentile=0.95)
diagnoser.train(train_data)
diagnoses = diagnoser.diagnose(test_data)

if not diagnoses.empty:
    print(f"Total diagnoses: {len(diagnoses)}")
    print("\nDiagnosis distribution:")
    display(diagnoses['diagnosis'].value_counts())
else:
    print("All measurements within normal range")

## 5. Outage Type Classifier

Decision-tree classifier combining ping, traceroute, and DNS signals. No training required — domain-knowledge-driven.

Categories: `HEALTHY`, `DNS_FAILURE`, `LOCAL_FIRST_MILE_OUTAGE`, `ROUTING_BLACKHOLE`, `ROUTING_LOOP`, `CONGESTION_AT_HOP`, `HIGH_LATENCY`, `MILD_DEGRADATION`, `PARTIAL_PACKET_LOSS`

In [ ]:
classifier = OutageClassifier()

sample_cases = [
    {"ping_success": True, "packet_loss": 0, "rtt_avg": 5.0},
    {"ping_success": True, "packet_loss": 0, "rtt_avg": 250.0},
    {"ping_success": True, "packet_loss": 33, "rtt_avg": 15.0},
    {"ping_success": False, "packet_loss": 100, "rtt_avg": 0},
]

print("Sample classifications:\n")
for case in sample_cases:
    classification, confidence, details = classifier.classify(case)
    print(f"  RTT={case.get('rtt_avg',0):.0f}ms, Loss={case.get('packet_loss',0)}%")
    print(f"    -> {classification} (confidence: {confidence:.0%})")
    print(f"       {details}\n")